# HW4: Adapting a Pretrained Model with LoRA

Run every cell from the top. **Everything already works.**

**Out:** Week 9, Class 1 · **Due:** Week 10, Class 1 · **100 points** · individual work

Work through the notebook and fill in each YOUR TURN cell. Submit this
`.ipynb` with all cells run and their output visible. There is no test to
pass: you are graded on the code working and on your short written answers.

Uses MiniLM, cached from Week 7. Everything runs on CPU in a few minutes.

Today you will:

1. Use a frozen pretrained model as a feature extractor.
2. Implement LoRA and count what you are actually training.
3. Measure the accuracy-versus-cost trade and argue for a setting.

There is no test to run and nothing to submit. Each task tells you what
you should see when it is right.

In [ ]:
# Setup.
import random
import numpy as np
import torch
import torch.nn as nn
import matplotlib.pyplot as plt
from transformers import AutoTokenizer, AutoModel, logging
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score

logging.set_verbosity_error()
torch.manual_seed(0); random.seed(0)

NAME = "sentence-transformers/all-MiniLM-L6-v2"
tok = AutoTokenizer.from_pretrained(NAME)
encoder = AutoModel.from_pretrained(NAME); encoder.eval()

GOOD = ["brilliant", "great", "superb", "wonderful", "clever", "funny"]
BAD = ["dull", "boring", "awful", "weak", "terrible", "lazy"]
NOUNS = ["film", "script", "cast", "acting", "story", "ending"]
texts, labels = [], []
for _ in range(400):
    positive = random.random() < 0.5
    w, n = random.choice(GOOD if positive else BAD), random.choice(NOUNS)
    texts.append(f"a {w} {n}"); labels.append(int(positive))

@torch.no_grad()
def embed(batch):
    ids = tok(batch, return_tensors="pt", padding=True, truncation=True)
    return encoder(**ids).last_hidden_state.mean(dim=1)

features = torch.cat([embed(texts[i:i+64]) for i in range(0, len(texts), 64)])
Xtr, Xte, ytr, yte = train_test_split(features.numpy(), labels, test_size=0.25,
                                      random_state=0, stratify=labels)
print("features:", features.shape)

## Part 1. Frozen features (25 points)

The cheapest adaptation of all: do not train the model at all.

In [ ]:
# ================== YOUR TURN 1 ==================
# Fit a logistic regression on the frozen features and report accuracy.
# Then report how many parameters you trained, against how many the
# encoder has.
#
# (25 points)
#
# Expected: accuracy near 1.00, training 385 numbers against the encoder's 22.7
#           million. That ratio is the whole argument for not fine-tuning first.
# ===============================================
clf = None          # <-- fit it

if clf is None:
    print("not yet")
else:
    acc = accuracy_score(yte, clf.predict(Xte))
    trained = clf.coef_.size + clf.intercept_.size
    frozen = sum(p.numel() for p in encoder.parameters())
    print(f"accuracy      : {acc:.3f}")
    print(f"you trained   : {trained:,} numbers")
    print(f"encoder holds : {frozen:,} numbers (all frozen)")
    print(f"ratio         : {frozen / trained:,.0f}x")

## Part 2. LoRA (45 points)

Now adapt a layer, without touching its weights.

In [ ]:
# ================== YOUR TURN 2 ==================
# Complete the LoRA layer. B must start at ZERO so the adapted model
# begins identical to the original.
#
# (25 points)
#
# Expected: with B at zero, the LoRA layer's output equals the base layer's
#           output exactly on the first forward pass. Check it before training.
# ===============================================
class LoRALayer(nn.Module):
    def __init__(self, base, rank=8, alpha=16):
        super().__init__()
        self.base = base
        for p in self.base.parameters():
            p.requires_grad = False
        self.A = None          # <-- shape (rank, in_features), small random
        self.B = None          # <-- shape (out_features, rank), ZEROS
        self.scale = alpha / rank

    def forward(self, h):
        return self.base(h)          # <-- add the low-rank term

base = nn.Linear(384, 2)
layer = LoRALayer(base, rank=8)
probe = torch.randn(4, 384)
same = torch.allclose(layer(probe), base(probe))
print("identical to the base layer before training:", same)
if layer.A is not None:
    print(f"trainable: {sum(p.numel() for p in layer.parameters() if p.requires_grad):,}")

In [ ]:
# ================== YOUR TURN 3 ==================
# Train the LoRA layer at ranks 1, 4 and 16. Plot accuracy against
# trainable parameters.
#
# (20 points)
#
# Expected: accuracy is high at every rank on a task this easy, while parameters
#           grow linearly with rank. The honest conclusion is that the smallest
#           rank you tried was already enough, and your report should say so.
# ===============================================
Xtr_t = torch.tensor(Xtr); ytr_t = torch.tensor(ytr)
Xte_t = torch.tensor(Xte); yte_t = torch.tensor(yte)

results = []
for rank in (1, 4, 16):
    torch.manual_seed(0)
    l = LoRALayer(nn.Linear(384, 2), rank=rank)
    params = [p for p in l.parameters() if p.requires_grad]
    if not params:
        print("not yet: finish LoRALayer first"); break
    opt = torch.optim.Adam(params, lr=0.05)
    for _ in range(150):
        opt.zero_grad()
        nn.functional.cross_entropy(l(Xtr_t), ytr_t).backward()
        opt.step()
    with torch.no_grad():
        acc = (l(Xte_t).argmax(1) == yte_t).float().mean().item()
    results.append((rank, sum(p.numel() for p in params), acc))
    print(f"   rank {rank:>2}: {results[-1][1]:>6,} params, accuracy {acc:.3f}")

## Part 3. Quantization and the write-up (30 points)

One more lever, then an argument.

In [ ]:
# ================== YOUR TURN 4 ==================
# Quantize the encoder's first layer weights to 8, 4 and 2 bits and
# measure the relative output error.
#
# (15 points)
#
# Expected: 8 bits costs well under 1%, 4 bits a few percent, 2 bits destroys
#           the layer. Memory falls linearly; quality does not.
# ===============================================
def quantize(t, bits):
    levels = 2 ** bits - 1
    lo, hi = t.min(), t.max()
    step = (hi - lo) / levels
    return torch.round((t - lo) / step) * step + lo

W = encoder.encoder.layer[0].attention.self.query.weight.data
probe = torch.randn(32, W.shape[1])
for bits in (8, 4, 2):
    # <-- compute the relative error of probe @ quantized(W).T against probe @ W.T
    print(f"   {bits} bits: (fill this in)")

In [ ]:
# ================== YOUR TURN 5 ==================
# You have 100 MB of storage per task and 20 minutes of CPU. Recommend
# a configuration and defend it with YOUR numbers.
#
# (15 points)
#
# Expected: there is no single right answer. Marks are for using the numbers you
#           measured, naming the trade you accepted, and saying what you would
#           measure next if you had more budget.
# ===============================================
# YOUR ANSWER (5 to 8 sentences). Cite the accuracy and parameter counts you
# measured above.
RECOMMENDATION = """
"""
print(RECOMMENDATION.strip() or "not yet")

## Answers

Try each task before reading.

In [ ]:
# Marking
#   Q1  25   logistic regression fitted; both parameter counts reported
#   Q2  25   LoRA correct, B initialised to zero, identity check passes
#   Q3  20   three ranks trained and compared
#   Q4  15   three bit-widths measured
#   Q5  15   a recommendation defended with your own numbers
#
# Common mistakes:
#   - initialising B randomly, so the adapted model starts BROKEN rather than
#     identical to the original, and the first epochs are spent recovering
#   - forgetting requires_grad = False on the base layer, which quietly trains
#     everything and defeats the point
#   - Q5 answered with a recommendation and no numbers